In [1]:
print('Installing zstd...')
!sudo apt-get update && sudo apt-get install -y zstd

Installing zstd...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,924 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadco

In [2]:
# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the server robustly in the background
import os
import time

print("Starting Ollama in the background...")
# nohup keeps the server running even after the cell finishes
os.system("nohup ollama serve > ollama_server.log 2>&1 &")

# Give the server 5 seconds to wake up
time.sleep(5)

# 3. Pull the model
print("Pulling the Llama 3 model...")
!ollama pull llama3
print("Ollama is ready!")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama in the background...
Pulling the Llama 3 model...

Ollama is ready!


In [ ]:


# ==========================================
# 2. THE PYTHON ROUTER SCRIPT
# ==========================================
import requests
import json
import pandas as pd

print("\n" + "="*40 + "\nStarting Python Evaluation...\n" + "="*40)

class LLMRouterV1:
    def __init__(self, model_name='llama3', taxonomy_path='/content/taxonomy_v1.json'):
        self.model_name = model_name
        self.api_url = "http://localhost:11434/api/generate"

        print(f"Loading taxonomy from {taxonomy_path}...")
        with open(taxonomy_path, 'r') as f:
            self.taxonomy = json.load(f)

    def build_system_prompt(self, domain):
        domain_key = domain.lower()
        if domain_key not in self.taxonomy['domains']:
            raise ValueError(f"Domain '{domain}' not found in taxonomy.")

        labels = self.taxonomy['domains'][domain_key]['labels']
        labels_text = "\n".join([f"- {l['name']}: {l['definition']}" for l in labels])

        system_prompt = f"""You are an expert routing agent for a {domain_key} support system.
Your task is to classify the user's request into EXACTLY ONE of the following routing categories:

{labels_text}

Analyze the user's prompt carefully. You must output your response ONLY as a valid JSON object with the following exact keys:
{{
    "predicted_label": "The exact name of the label from the list above",
    "confidence_level": "High, Medium, or Low",
    "short_reason": "One short sentence explaining why you chose this label",
    "needs_clarification": true or false
}}
Do not include any markdown formatting, conversational text, or explanations outside of the JSON object.
"""
        return system_prompt

    def route_request(self, user_prompt, domain):
        system_prompt = self.build_system_prompt(domain)
        full_prompt = f"{system_prompt}\n\nUSER REQUEST:\n\"{user_prompt}\""

        payload = {
            "model": self.model_name,
            "prompt": full_prompt,
            "stream": False,
            "format": "json"
        }

        try:
            response = requests.post(self.api_url, json=payload)
            response.raise_for_status()
            result_text = response.json().get("response", "{}")
            return json.loads(result_text)

        except Exception as e:
            print(f"  -> Error for prompt '{user_prompt[:30]}...': {e}")
            return {
                "predicted_label": "Error",
                "confidence_level": "Low",
                "short_reason": f"API Error",
                "needs_clarification": True
            }

    def evaluate_benchmark(self, input_csv, output_csv):
        print(f"Loading benchmark data from {input_csv}...")
        df = pd.read_csv(input_csv)
        results = []

        print(f"Routing {len(df)} requests. This will take a few minutes...")

        for index, row in df.iterrows():
            prompt_text = row['prompt']
            domain = row['domain']

            if index % 10 == 0:
                 print(f"Processing [{index}/{len(df)}]...")

            llm_output = self.route_request(prompt_text, domain)

            result_row = {
                "prompt_id": row.get('prompt_id', f"ID-{index}"),
                "domain": domain,
                "user_prompt": prompt_text,
                "gold_label": row.get('label', ''),
                "is_ambiguous_gold": row.get('is_ambiguous', ''),
                "predicted_label": llm_output.get("predicted_label", ""),
                "confidence_level": llm_output.get("confidence_level", ""),
                "needs_clarification_pred": llm_output.get("needs_clarification", False),
                "short_reason": llm_output.get("short_reason", "")
            }
            results.append(result_row)

        results_df = pd.DataFrame(results)
        results_df.to_csv(output_csv, index=False)
        print(f"\nDone! Results saved to {output_csv}")

        correct = (results_df['gold_label'] == results_df['predicted_label']).sum()
        total = len(results_df)
        print(f"Rough Accuracy: {correct}/{total} ({(correct/total)*100:.1f}%)")

# ==========================================
# 3. RUN THE PIPELINE
# ==========================================
taxonomy_path = "/content/taxonomy_v1.json"
benchmark_path = "/content/v0_pilot_benchmark.csv"
output_path = "/content/v1_llm_results.csv"

# Start the router and run it!
router = LLMRouterV1(model_name='llama3', taxonomy_path=taxonomy_path)
router.evaluate_benchmark(benchmark_path, output_path)


Starting Python Evaluation...
Loading taxonomy from /content/taxonomy_v1.json...
Loading benchmark data from /content/v0_pilot_benchmark.csv...
Routing 138 requests. This will take a few minutes...
Processing [0/138]...
